# CNN1D — intra-version et inter-version, avec et sans couche dense

Notebook ayant produit les résultats du dossier `results/cnn1d_*` (trois cas par système).

> **Exécution** : notebook conçu pour Google Colab. Les chemins pointent vers Google Drive (`/content/drive/MyDrive/bug/...`). Voir [`data/README.md`](../../data/README.md) pour obtenir les données et adapter les chemins.

*Nom d'origine dans Colab : `Untitled79`*


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
df_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-3.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.7.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',


]

# New Section

In [ ]:
import pandas as pd

# Chargement des DataFrames
df_paths = [
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv', usecols=lambda x: x != 'name')

dfs = [pd.read_csv(path, usecols=lambda x: x != 'name') for path in df_paths]

# Calcul de la moyenne et de l'écart-type pour df_val
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5 pour df_val
cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Normaliser df_val
cols_to_keep_val = df_val.columns[:-1]
means_val = means_val[cols_to_keep_val]
stds_val = stds_val[cols_to_keep_val]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val) / stds_val

remaining_columns_val = len(cols_to_keep_val)
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")

# Calcul de la moyenne et de l'écart-type pour chaque DataFrame dans dfs
means = []
stds = []
for i, df in enumerate(dfs):
    df_without_last_col = df.iloc[:, :-1]
    means.append(df_without_last_col.mean(axis=0))
    stds.append(df_without_last_col.std(axis=0))

# Standardisation des DataFrames et suppression des colonnes avec std > 1e-5
standardized_dfs = []
for i, df in enumerate(dfs):
    df_without_last_col = df.iloc[:, :-1]

    # Supprimer les colonnes avec std > 1e-5
    cols_to_drop = stds[i][abs(stds[i]) <= 1e-5].index.values
    df = df.drop(cols_to_drop, axis=1)

    # Normaliser les colonnes restantes
    cols_to_keep = df.columns[:-1]
    means[i] = means[i][cols_to_keep]
    stds[i] = stds[i][cols_to_keep]
    df[cols_to_keep] = (df[cols_to_keep] - means[i]) / stds[i]

    # Ajout de la dernière colonne non modifiée
    last_col = df.iloc[:, -1]
    df_standardized = pd.concat([df[cols_to_keep], last_col], axis=1)
    standardized_dfs.append(df_standardized)

    remaining_columns_training = len(cols_to_keep)
    print(f"Nombre de colonnes restantes dans le DataFrame df_v{i} : {remaining_columns_training}")

# Affichage des DataFrames standardisés
for i, df in enumerate(standardized_dfs):
    print(f'DataFrame df_v{i} standardisé:')
    print(df)
    print()

# Concaténation des DataFrames standardisés sauf df_val
df_train = pd.concat(standardized_dfs, ignore_index=True)

# Affichage du DataFrame concaténé
print(df_train)
print(df_val)


# New Section

In [ ]:
import pandas as pd

# Liste des chemins de fichiers CSV pour l'entrainement
train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv', usecols=lambda x: x != 'name')

# Boucle pour lire les fichiers CSV
train_dfs = []
for file_path in train_file_paths:
    df = pd.read_csv(file_path, usecols=lambda x: x!= 'name')
    train_dfs.append(df)

# Concaténation des datasets 'camel' en un seul DataFrame
df_train = pd.concat(train_dfs, ignore_index=True)

# Affichage des DataFrames concaténés
print(df_train)
print(df_val)


# New Section

In [ ]:
# Fonction pour convertir les valeurs en binaire
def convert_to_binary(value):
    return 1 if value > 0 else 0

# Appliquer la fonction de conversion à la colonne 'bug'
df_train['bug'] = df_train['bug'].apply(convert_to_binary)
df_val['bug'] = df_val['bug'].apply(convert_to_binary)

# Afficher le dataset mis à jour
print(df_train)
print(df_val)
print(df_train.describe())


In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

# Use SMOTE to oversample the minority class in X_train
smote = SMOTE(sampling_strategy='auto')

# Resample X_train and y_train
X_train_resampled, y_train_resampled = smote.fit_resample(df_train.iloc[:, :-1].values, df_train.iloc[:, -1:].values)


x_train = pd.DataFrame(X_train_resampled, columns=df_train.columns[: -1])
y_train = pd.DataFrame(y_train_resampled, columns=df_train.columns[-1:])

df_training = pd.concat([x_train, y_train], axis=1)

print(df_training)


In [ ]:
import pandas as pd

# Calculer les moyennes et écarts types pour l'ensemble d'entraînement
means_tr = df_training[df_training.columns[:-1]].mean(axis=0)
stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

# Calculer les moyennes et écarts types pour l'ensemble de validation
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5
cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-4].index.values
cols_to_drop_val = stds_val[abs(stds_val) <= 1e-4].index.values

df_training = df_training.drop(cols_to_drop_training, axis=1)
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Sélectionner les colonnes restantes
cols_to_keep_training = df_training.columns[:-1]
cols_to_keep_val = df_val.columns[:-1]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

# Normaliser les colonnes restantes
df_training[cols_to_keep_training] = (df_training[cols_to_keep_training] - means_tr[cols_to_keep_training]) / stds_tr[cols_to_keep_training]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val[cols_to_keep_val]) / stds_val[cols_to_keep_val]

print(df_training)
print(df_val)

print(f"Nombre de colonnes restantes dans l'ensemble d'entraînement : {remaining_columns_training}")
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")


In [ ]:

train = pd.get_dummies(df_training, dtype= int, columns=['bug'], prefix=['bug_'])
val = pd.get_dummies(df_val, dtype= int, columns=['bug'], prefix=['bug_'])

print(train.describe())

x_training = train.iloc[:, :-2]
y_training = train.iloc[:, -2:]

#print(x_training)

x_val = val.iloc[:, :-2]
y_val = val.iloc[:, -2:]

print(val.describe())

In [ ]:
x_training = x_training.values
y_training = y_training.values
x_val =  x_val.values
y_val =  y_val.values


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(16, 2, activation='relu', input_shape=(20, 1)),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.22191711633138964),
    tf.keras.layers.Conv1D(64, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.22191711633138964),
    tf.keras.layers.Conv1D(256, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.22191711633138964),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.00512825663132412)),  # Ajout de la régularisation L2
    tf.keras.layers.Dense(2, activation='softmax', kernel_regularizer=regularizers.l2(0.00512825663132412))  # Ajout de la régularisation L2
])
model.summary()


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Compiler le modèle
model.compile(optimizer=Adam(learning_rate=0.000464950189674214),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Configuration de l'arrêt précoce
early_stopping = EarlyStopping(monitor='val_loss', patience=450, restore_best_weights=True)

# Entraîner le modèle avec l'arrêt précoce
history = model.fit(x_training, y_training, epochs=500, batch_size=8, validation_data=(x_val, y_val), callbacks=[early_stopping])


In [ ]:
import matplotlib.pyplot as plt

# Graphique de l'accuracy
plt.figure(figsize=(4, 2))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Graphique de la loss
plt.figure(figsize=(4, 2))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.metrics import geometric_mean_score

# Prédire les valeurs de sortie pour x_val
y_pred = model.predict(x_val)

# Convertir les prédictions en classes
y_pred_classes = y_pred.argmax(axis=1)

# Convertir les étiquettes en classes
y_test_classes = y_val.argmax(axis=1)

# Calculer la matrice de confusion
cm = confusion_matrix(y_test_classes, y_pred_classes)

print(cm)

# Calculer le G-mean directement
g_mean = geometric_mean_score(y_test_classes, y_pred_classes)
print(f'G-mean: {g_mean}')

# Visualiser la matrice de confusion
plt.figure(figsize=(5, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Pas Bug', 'Bug'], yticklabels=['Pas Bug', 'Bug'])
plt.xlabel('Prédictions')
plt.ylabel('Vraies Classes')
plt.title('Matrice de Confusion')
plt.show()


In [ ]:
from sklearn.metrics import classification_report

# Faire des prédictions
predictions = model.predict(x_val)

# Convertir les prédictions en classes
predictions_binary = predictions.argmax(axis=1)

# Convertir les cibles en classes
y_val_classes = y_val.argmax(axis=1)

# Générer le rapport de classification
report = classification_report(y_val_classes, predictions_binary)

# Afficher le rapport de classification
print("Classification Report:")
print(report)


In [ ]:
from sklearn.model_selection import KFold
import numpy as np


losses = []
accuracies = []

kfold = KFold(n_splits=15, shuffle=True, random_state=42)

for train_index, val_index in kfold.split(x_training):
    x_train_fold, X_val_fold = x_training[train_index], x_training[val_index]
    y_train_fold, y_val_fold = y_training[train_index], y_training[val_index]

    model.fit(x_train_fold, y_train_fold, epochs=50, batch_size=64, verbose=0)
    loss, accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    losses.append(loss)
    accuracies.append(accuracy)
    print(f"Loss: {loss}, Accuracy: {accuracy}")

# Calculer la moyenne et l'écart-type
mean_loss = np.mean(losses)
std_loss = np.std(losses)
mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

print(f"Mean Loss: {mean_loss}, Standard Deviation of Loss: {std_loss}")
print(f"Mean Accuracy: {mean_accuracy}, Standard Deviation of Accuracy: {std_accuracy}")